# Clima y demanda del subte de Buenos Aires (2018–2021): reproducción en Colab

Trabajo Final Integrador, Metodología de la Investigación, Grupo 36.

Ejecutá las celdas en orden con *Entorno de ejecución → Ejecutar todas*. No hace falta cuenta de GitHub ni de Google para los datos. La corrida completa tarda unos 15 a 20 minutos; el paso más largo es el script 03, que procesa unos 25 millones de filas.

Las versiones de Colab difieren de las originales (Python 3.8, pandas 2.0.3, numpy 1.24); pueden aparecer diferencias numéricas menores, que la celda 6 permite ver.

In [ ]:
# 1. Clonar el repositorio (público, sin credenciales)
REPO_URL = "https://github.com/leonnf/clima-demanda-subte-bsas.git"
RAIZ = "/content/clima-demanda-subte-bsas"

import os
if not os.path.exists(RAIZ):
    !git clone -q {REPO_URL} {RAIZ}
%cd {RAIZ}
!git log --oneline -1

In [ ]:
# 2. Dependencias
# Colab ya trae numpy, scipy, statsmodels, scikit-learn, matplotlib y pyarrow.
# Solo aseguramos pandas 2.x (con pandas 3 cambian levemente los días atípicos)
# y las librerías que no vienen preinstaladas. No se fija numpy: numpy 1.x no tiene
# versión compilada para el Python de Colab (3.13) y pip intentaría compilarlo.
!pip -q install "pandas>=2.2,<3" holidays gdown statsmodels scikit-learn pyarrow

import sys, numpy, pandas, sklearn, statsmodels
print("Python", sys.version.split()[0], "| numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| scikit-learn", sklearn.__version__, "| statsmodels", statsmodels.__version__)

In [ ]:
# 3. Datos crudos (dsets/). El clima y la demanda diaria ya vienen en recursos/.
# Por defecto se descarga dsets.zip del Drive del grupo (público, solo lectura).
# Alternativa: FUENTE = "catedra" baja la carpeta original de la cátedra.
FUENTE = "espejo_grupo"
ID_ZIP_GRUPO = "1Oe0TMeqHbQueOMjrGGuSEbe4SOoa42Bg"
ID_CARPETA_CATEDRA = "1N_upx66kxNa6MqUD40SsvJqioQLJ8jdm"

import os, glob, shutil
if os.path.exists("dsets/historico_2019.csv"):
    print("dsets/ ya está presente, no se descarga.")
elif FUENTE == "espejo_grupo":
    !gdown {ID_ZIP_GRUPO} -O dsets.zip
    !unzip -q -o dsets.zip && rm -f dsets.zip
    # Si el zip no traía la carpeta dsets/, mover los archivos sueltos a dsets/
    if not os.path.exists("dsets/historico_2019.csv"):
        os.makedirs("dsets", exist_ok=True)
        for f in glob.glob("*.csv") + glob.glob("*.xlsx"):
            shutil.move(f, "dsets/")
else:
    !gdown --folder {ID_CARPETA_CATEDRA} -O dsets

!ls -lh dsets/

In [ ]:
# 4. Verificar los datos contra el manifiesto (SHA-256)
# Los archivos de dsets/ deben dar OK. Los Parquet de recursos/ aparecen como FALTA
# hasta que se ejecutan los scripts 02 y 03 (celda 5): es lo esperado.
!python descargar_datos.py --solo-verificar

In [ ]:
# 5. Ejecutar el pipeline completo, en orden
import subprocess, sys, time
SCRIPTS = [
    "02_base_analitica.py",
    "03_molinetes_2018_2019.py",
    "03b_verificacion_2020_2021.py",
    "03c_explicacion_diferencias.py",
    "04_eda.py",
    "05_hipotesis.py",
    "06_segmentacion_H3.py",
    "07_modelo_predictivo.py",
    "09_figuras.py",
]
total = time.time()
for s in SCRIPTS:
    t = time.time()
    r = subprocess.run([sys.executable, f"Ultima entrega/notebooks/{s}"], capture_output=True, text=True)
    print(f"{s:34s} rc={r.returncode}  {time.time() - t:6.0f} s")
    if r.returncode != 0:
        print(r.stdout[-1500:]); print(r.stderr[-3000:])
        raise SystemExit(f"Falló {s}")
print(f"Total: {(time.time() - total) / 60:.1f} min")

In [ ]:
# 6. Comparar con los resultados publicados en el repositorio
# git diff muestra qué tablas cambiaron respecto de la versión del informe.
# Con otras versiones de pandas pueden variar levemente los días atípicos (ver README).
!git diff --stat -- "Ultima entrega/resultados/tablas"

import pandas as pd
pd.set_option("display.width", 200)
T = "Ultima entrega/resultados/tablas/"
for nombre in ["t05_especificaciones", "t05_h2_contraste", "t06_kmeans_diagnostico", "t07_metricas"]:
    print(f"\n=== {nombre} ===")
    display(pd.read_csv(T + nombre + ".csv"))

In [ ]:
# 7. Figuras principales
from IPython.display import Image, display
F = "Ultima entrega/resultados/figuras/"
for f in ["fig01_serie_diaria_fases_atipicos.png", "fig05_boxplots_h2.png",
          "fig06_forest_plot_especificaciones.png", "fig10_perfiles_por_cluster.png",
          "fig12_real_vs_predicho_rmse.png"]:
    print(f); display(Image(F + f, width=900))